# Orion Agent en Google Colab

Agente autónomo con un LLM local (**Ollama**), memoria propia y control de su código. Todo corre en la máquina virtual de Colab.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. Sin GPU también funciona, pero mucho más lento.

> ⚠️ El agente puede ejecutar comandos, descargar y subir archivos. Por defecto **pide tu permiso** para lo sensible (`shell`, `upload`, cambios a su propio código). Colab es una máquina desechable: no guardes ahí credenciales reales.

## 1. Descargar el proyecto y comprobar el hardware

In [ ]:
!git clone -q https://github.com/Miguel283332/Orion.git 2>/dev/null || (cd Orion && git pull -q)
%cd Orion
!nvidia-smi -L || echo "Sin GPU NVIDIA: se usará la CPU"

## 2. Dónde guardar la memoria

Si montas Google Drive, la memoria, el estado y los modelos afinados **sobreviven** a que se cierre la sesión. Si no, se pierden al desconectarte.

In [ ]:
USAR_DRIVE = True   # pon False para no usar Drive

import os
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    HOME = '/content/drive/MyDrive/orion_home'
else:
    HOME = '/content/orion_home'
os.makedirs(HOME, exist_ok=True)
os.environ['ORION_HOME'] = HOME
print('Memoria y estado en:', HOME)

## 3. Instalar Ollama y descargar el modelo

La primera vez tarda unos minutos. Puedes cambiar `MODELO` por otro de la [biblioteca de Ollama](https://ollama.com/library) (con la GPU T4 de Colab van bien los de 3B–8B).

In [ ]:
MODELO = 'llama3.2:3b'
!python orion_agent.py --model {MODELO} setup

## 4. Pruebas internas (opcional, ~30 s)

Comprueban permisos, memoria, red y la compuerta de automodificación.

In [ ]:
!python orion_agent.py selftest --quiet

## 5. Hablar con el agente

Escribe un mensaje; el agente razona y usa herramientas. Cuando quiera hacer algo sensible, te preguntará aquí mismo.
Escribe `salir` para terminar.

> **Si el agente se parchea a sí mismo** (`self_patch`), el código nuevo queda guardado en `orion_agent.py` pero esta sesión sigue usando el anterior. Para cargarlo: `import importlib; importlib.reload(oa)` y vuelve a crear `agent`. (En el modo continuo `run` el reinicio es automático.)


In [ ]:
import sys
sys.path.insert(0, '.')
import orion_agent as oa

args = oa.build_parser().parse_args(['--home', HOME, '--model', MODELO, '--autonomy', 'supervised', 'chat'])
agent = oa.make_agent(args, interactive=True)   # interactive=True: los permisos se piden con input()

while True:
    texto = input('\ntú> ').strip()
    if texto.lower() in ('salir', 'exit', '/salir'):
        break
    if texto:
        print('\norion>', agent.chat_turn(texto))

## 6. Modo continuo (en segundo plano)

El agente ejecuta los objetivos que le dejes en la bandeja, reflexiona sobre cada uno y guarda lo aprendido.
`--free` hace que, sin tareas, él mismo elija qué hacer (ordenar su memoria, practicar…).

In [ ]:
!nohup python orion_agent.py --home {HOME} --model {MODELO} run --free --interval 10 > run.log 2>&1 &
!python orion_agent.py --home {HOME} goal "Resume en tu memoria núcleo quién eres y qué herramientas tienes."

In [ ]:
# ver qué hace
!tail -n 25 run.log

Si el agente pide un permiso mientras corre en segundo plano (no hay terminal), lo verás en `run.log` y aquí:

In [ ]:
!python orion_agent.py --home {HOME} approvals
# aprobar o denegar:  !python orion_agent.py --home {HOME} approve ID    /    deny ID

## 7. Ver la memoria y parar el agente

In [ ]:
!python orion_agent.py --home {HOME} memory
!python orion_agent.py --home {HOME} stop

## 8. (Experimental) Consolidar la experiencia en los pesos del modelo

El agente guarda las trayectorias que él mismo evalúa como buenas. Con `sleep` las usa para afinar un adaptador **LoRA**, lo registra en Ollama y **solo lo adopta si mejora** en las tareas de `eval`.
Necesita GPU y bastantes ejemplos (≥30 pasos buenos). Está escrito pero **no se ha probado con una GPU real**; usa Drive para no perder el resultado.

In [ ]:
!pip -q install transformers peft accelerate bitsandbytes
!python orion_agent.py --home {HOME} --model {MODELO} eval

In [ ]:
!python orion_agent.py --home {HOME} --model {MODELO} sleep